In [0]:
%pip install faker
# %restart_python

import uuid
import random
from datetime import datetime, timedelta
import faker

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
fake = faker.Faker()

# Load dimension sample data from Delta tables (not Parquet now)
customer_ids = [row.customer_id for row in spark.sql("SELECT customer_id FROM main.ordersystem_sample.customers").collect()]
store_ids = [row.store_id for row in spark.sql("SELECT store_id FROM main.ordersystem_sample.stores").collect()]
product_ids = [row.product_id for row in spark.sql("SELECT product_id FROM main.ordersystem_sample.products").collect()]

# Enums
order_status_choices = ['PENDING', 'SHIPPED', 'DELIVERED', 'CANCELLED', 'COMPLETED']
item_status_choices = ['PENDING', 'ALLOCATED', 'SHIPPED', 'DELIVERED', 'CANCELLED']

# ----------------------------
# Insert Orders and Line Items
# ----------------------------
for _ in range(1000):
    order_id = str(uuid.uuid4())
    customer_id = random.choice(customer_ids)
    store_id = random.choice(store_ids)
    order_status = random.choice(order_status_choices)
    created_days_ago = random.randint(0, 90)
    created_at = (datetime.now() - timedelta(days=created_days_ago)).isoformat()

    # Insert single order row
    spark.sql(f"""
        INSERT INTO main.ordersystem_sample.orders
        VALUES (
            '{order_id}',
            '{customer_id}',
            '{store_id}',
            '{order_status}',
            TIMESTAMP('{created_at}'),
            TIMESTAMP('{created_at}')
        )
    """)

    # Generate 1–5 line items per order
    for _ in range(random.randint(1, 5)):
        line_item_id = str(uuid.uuid4())
        product_id = random.choice(product_ids)
        quantity = random.randint(1, 10)
        item_status = random.choice(item_status_choices)

        spark.sql(f"""
            INSERT INTO main.ordersystem_sample.order_line_items
            VALUES (
                '{line_item_id}',
                '{order_id}',
                '{product_id}',
                {quantity},
                '{item_status}',
                TIMESTAMP('{created_at}')
            )
        """)